In [1]:
import os
import face_recognition
import pickle

def encode_faces(dataset_path):
    """Extract face encodings from images in the dataset."""
    encodings = []
    names = []

    # Loop through each subfolder in the dataset directory
    for student_id in os.listdir(dataset_path):
        student_folder = os.path.join(dataset_path, student_id)

        # Check if it's a directory (to avoid files)
        if os.path.isdir(student_folder):
            # Loop through each image in the student's folder
            for image_name in os.listdir(student_folder):
                if image_name.endswith(".jpg") or image_name.endswith(".png"):
                    image_path = os.path.join(student_folder, image_name)
                    # Load the image using face_recognition
                    image = face_recognition.load_image_file(image_path)
                    # Get face encodings for the loaded image
                    face_encodings = face_recognition.face_encodings(image)

                    # If a face encoding was found, append it to our list
                    if face_encodings:
                        encodings.append(face_encodings[0])  # Take the first encoding
                        names.append(student_id)  # Use the student ID as the name

    # Serialize the encodings and names to disk
    data = {"encodings": encodings, "names": names}
    with open("encodings.pickle", "wb") as f:
        f.write(pickle.dumps(data))

# Example usage
encode_faces("TrainingImage")


In [2]:
import os
import pickle

def load_encodings(encodings_file):
    """Load face encodings from a pickle file."""
    if os.path.exists(encodings_file):
        with open(encodings_file, "rb") as f:
            data = pickle.loads(f.read())
            return data["encodings"], data["names"]
    else:
        return None, None

def main():
    """Main function to run the automatic attendance system."""
    
    subject_name = input("Enter Subject Name for Attendance: ")
    
    # Load previously saved encodings if they exist
    encodings_file = "encodings.pickle"
    print("Loading training data...")
    
    encodings, names = load_encodings(encodings_file)
    
    if encodings is None or names is None:
        print("No saved encodings found, training new data...")
        encode_faces("TrainingImages")  # Train and save new encodings
        encodings, names = load_encodings(encodings_file)  # Load newly trained data
    
    print("Testing model...")
    test_model(encodings)  # Call your test_model function here

if __name__ == "__main__":
    main()


Enter Subject Name for Attendance:  Science


Loading training data...
Testing model...


NameError: name 'test_model' is not defined

## Updated Model

In [ ]:
import os
import cv2
import datetime
import dlib
import face_recognition
import numpy as np
import mysql.connector
import pickle
import csv

def connect_to_database():
    """Connect to the MySQL database."""
    try:
        connection = mysql.connector.connect(
            host='localhost',  # Change if necessary
            user='root',  # Replace with your MySQL username
            password='ExplorerSoul@2024',  # Replace with your MySQL password
            database='attendance_management_system'  # Replace with your database name
        )
        return connection
    except mysql.connector.Error as err:
        print(f"Error: {err}")
        return None

def encode_faces(dataset_path):
    """Extract face encodings from images in the dataset."""
    encodings = []
    names = []

    # Loop through each subfolder in the dataset directory
    for student_id in os.listdir(dataset_path):
        student_folder = os.path.join(dataset_path, student_id)

        # Check if it's a directory (to avoid files)
        if os.path.isdir(student_folder):
            # Loop through each image in the student's folder
            for image_name in os.listdir(student_folder):
                if image_name.endswith(".jpg") or image_name.endswith(".png"):
                    image_path = os.path.join(student_folder, image_name)
                    # Load the image using face_recognition
                    try:
                        image = face_recognition.load_image_file(image_path)
                        # Get face encodings for the loaded image
                        face_encodings = face_recognition.face_encodings(image)

                        # If a face encoding was found, append it to our list
                        if face_encodings:
                            encodings.append(face_encodings[0])  # Take the first encoding
                            names.append(student_id)  # Use the student ID as the name

                    except Exception as e:
                        print(f"An error occurred while processing {image_path}: {e}")

    # Serialize the encodings and names to disk
    data = {"encodings": encodings, "names": names}
    with open("encodings.pickle", "wb") as f:
        f.write(pickle.dumps(data))

def load_encodings(encodings_file):
    """Load face encodings from a pickle file."""
    if os.path.exists(encodings_file):
        with open(encodings_file, "rb") as f:
            data = pickle.loads(f.read())
            return data["encodings"], data["names"]
    else:
        return None, None

def take_img(enrollment, name):
    """Capture images from webcam and save them in a specified directory."""
    if enrollment == '' or name == '':
        print("Please fill in all fields!")
        return
    
    try:
        base_dir = 'TrainingImage'
        if not os.path.exists(base_dir):
            os.makedirs(base_dir)

        enrollment_dir = os.path.join(base_dir, enrollment)
        if not os.path.exists(enrollment_dir):
            os.makedirs(enrollment_dir)

        cam = cv2.VideoCapture(0)
        detector = dlib.get_frontal_face_detector()
        
        sampleNum = 0
        
        print("Starting image capture. Press 'q' to quit.")
        
        while True:
            ret, img = cam.read()
            
            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            faces = detector(rgb_img)
            
            for face in faces:
                x, y, w, h = (face.left(), face.top(), face.width(), face.height())
                
                face_image = rgb_img[y:y+h, x:x+w]
                
                if face_image.ndim != 3 or face_image.shape[2] != 3:
                    print("Captured image is not in RGB format. Skipping this capture.")
                    continue
                
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
                sampleNum += 1
                
                cv2.imwrite(f"{enrollment_dir}/{name}.{enrollment}.{sampleNum}.jpg", face_image)
                print(f"Image {sampleNum} saved for Enrollment: {enrollment}, Name: {name}")
                
            cv2.imshow('Frame', img)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
            if sampleNum >= 140:  # Capture up to 140 images per student.
                break

        cam.release()
        cv2.destroyAllWindows()

    except Exception as e:
        print(f"An error occurred: {e}")

def prepare_training_data(data_folder_path):
    """Prepare training data by loading images and extracting face encodings."""
    encodings = []
    labels = []
    
    for dir_name in os.listdir(data_folder_path):
        if not dir_name.startswith('.'):
            label = dir_name  # Assuming dir_name is the enrollment ID
            subject_dir_path = os.path.join(data_folder_path, dir_name)
            
            for image_name in os.listdir(subject_dir_path):
                if image_name.endswith(".jpg"):
                    image_path = os.path.join(subject_dir_path, image_name)
                    try:
                        image = face_recognition.load_image_file(image_path)
                        encoding = face_recognition.face_encodings(image)

                        if encoding:
                            encodings.append(encoding[0])  # Take the first encoding
                            labels.append(label)  # Use the student ID as the name

                    except Exception as e:
                        print(f"An error occurred while processing {image_path}: {e}")
    
    return encodings, labels

def test_model(encodings, names):
    """Test the model by capturing images from webcam and recognizing faces."""
    cam = cv2.VideoCapture(0)
    
    while True:
        ret, img = cam.read()
        
        if not ret:
            print("Failed to capture image from webcam.")
            break
        
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        face_locations = face_recognition.face_locations(rgb_img)
        face_encodings = face_recognition.face_encodings(rgb_img, face_locations)
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            matches = face_recognition.compare_faces(encodings, face_encoding)
            
            label_text = "Unknown"
            
            if True in matches:
                first_match_index = matches.index(True)
                label_text = names[first_match_index]  # Use actual student ID
                
                # Record attendance in the database and CSV file
                record_attendance(label_text, label_text)  # Store both ID and Name

            cv2.rectangle(img, (left, top), (right, bottom), (255, 0, 0), 2)
            cv2.putText(img, label_text, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        cv2.imshow('Face Recognition', img)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cam.release()
    cv2.destroyAllWindows()

def record_attendance(enrollment_id, name):
    """Record attendance in the MySQL database and save it to a CSV file."""
    date_today = datetime.datetime.now().date()  # Get today's date
    time_now = datetime.datetime.now().time()     # Get current time
    
    connection = connect_to_database()
    
    if connection:
        cursor = connection.cursor()
        
        # Use INSERT ... ON DUPLICATE KEY UPDATE to handle duplicates
        sql_query = """
        INSERT INTO attendance (id, name, date, time) 
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE time=VALUES(time);
        """
        
        cursor.execute(sql_query,
                       (enrollment_id,
                        name,
                        date_today.strftime('%Y-%m-%d'),
                        time_now.strftime('%H:%M:%S')))
        
        connection.commit()  # Commit changes to the database
        
        cursor.close()
        connection.close()
        
        print(f"Attendance recorded for Student ID: {enrollment_id} ({name}) on {date_today} at {time_now}")

    # Save attendance to CSV file based on subject name
    save_attendance_to_csv(enrollment_id, name)

def save_attendance_to_csv(enrollment_id, name):
    """Save attendance record to a CSV file based on subject."""
    subject_name = input("Enter Subject Name for Attendance: ")   # Ask for subject name each time attendance is recorded.
    
    attendance_folder = 'attendance'
    
    # Create attendance folder if it doesn't exist
    if not os.path.exists(attendance_folder):
        os.makedirs(attendance_folder)

    csv_file_path = os.path.join(attendance_folder, f'attendance_{subject_name}.csv')

    # Check if the CSV file exists to determine whether to write headers or not
    file_exists = os.path.isfile(csv_file_path)

    with open(csv_file_path, mode='a', newline='') as csvfile:
        fieldnames = ['ID', 'Name', 'Date', 'Time']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # Write header only once
        if not file_exists:
            writer.writeheader()

        writer.writerow({
            'ID': enrollment_id,
            'Name': name,
            'Date': datetime.datetime.now().strftime('%Y-%m-%d'),
            'Time': datetime.datetime.now().strftime('%H:%M:%S')
        })

def main():
    """Main function to run the automatic attendance system."""
    
    subject_name = input("Enter Subject Name for Attendance: ")
    
    # Load previously saved encodings if they exist
    encodings_file = "encodings.pickle"
    
    print("Loading training data...")
    
    encodings, names = load_encodings(encodings_file)
    
    if encodings is None or names is None:
        print("No saved encodings found. Training new data...")
        encode_faces("TrainingImages")  # Train and save new encodings
        encodings, names = load_encodings(encodings_file)  # Load newly trained data
    
    print("Testing model...")
    test_model(encodings,names)  # Call your test_model function here

if __name__ == "__main__":
   main()


Enter Subject Name for Attendance:  Biology


Loading training data...
Testing model...
Attendance recorded for Student ID: 1 (1) on 2024-12-11 at 19:37:26.736097


Enter Subject Name for Attendance:  q
